# BC카드 공모전 — 5단계 외부데이터 수집 (한국관광 데이터랩 방문자수)

구글트렌드는 후보지 대부분이 소규모 시군구라 검색량 데이터가 너무 희박해서 못 썼다(9/17). 대신 **한국관광공사 TourAPI(관광빅데이터) — 기초 지자체 지역방문자수 API**를 쓴다.

이 API는 KT(내국인)·SKT(외국인) 통신데이터 기반 일별 방문자수를 시군구 단위로 제공한다. `touDivCd=3`이 외국인 방문자수라서, 우리 카드데이터(GENDER_CD=3)와 **완전히 다른 원천으로 같은 대상을 측정**할 수 있다 — 카드소비 증가 시점과 실제 통신데이터 기반 방문자 증가 시점이 겹치는지 교차검증하는 용도.

**주의**: 이 API는 지역 필터 파라미터가 없다(공식 매뉴얼에도 없고, 실제로 signguCd 등을 넣으면 에러 남) — 날짜만 넣으면 전국 시군구×관광객구분 전체가 통째로 나온다. 그래서 우리 후보지 8개는 받은 다음에 코드로 걸러낸다.

**실행 전 필요한 것**: data.go.kr에서 발급받은 서비스키(인증키) — 발급받은 키를 아래 SERVICE_KEY에 넣어서 실행한다 (제출·공유용 파일에는 키를 남기지 않았다).

## 0. 환경 설정

In [ ]:
import requests
import pandas as pd
import time

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 150)

from google.colab import drive
drive.mount('/content/drive')
BASE_DIR = '/content/drive/MyDrive/BC카드'

# 키는 코드에 남기지 말고 실행할 때만 넣는다 (공유·제출 전에는 지운다)
# data.go.kr에서 발급받은 일반 인증키 (Decoding 키 — requests가 자동으로 URL 인코딩해주니 그대로 사용)
SERVICE_KEY = '여기에_data.go.kr_서비스키_입력'

API_URL = 'http://apis.data.go.kr/B551011/DataLabService/locgoRegnVisitrDDList'


## 1. 후보지 시군구코드 정의

공식 법정동코드 기준(행정안전부). 고성군은 강원/경남에 동명 지역이 있어서 코드로 구분해야 한다 — 이번 8개 후보지 중에는 강원 고성군(42820)만 해당.

In [ ]:
TARGET_REGIONS = {
    '47750': ('경상북도', '청송군'),
    '47920': ('경상북도', '봉화군'),
    '47760': ('경상북도', '영양군'),
    '47730': ('경상북도', '의성군'),
    '42820': ('강원특별자치도', '고성군'),
    '42750': ('강원특별자치도', '영월군'),
    '42830': ('강원특별자치도', '양양군'),
    '28720': ('인천광역시', '옹진군'),
}

FOREIGNER_CODE = '3'  # touDivCd: 1=현지인, 2=외지인, 3=외국인


## 2. 사전 점검 — 날짜 범위 쿼리가 실제로 여러 날을 한번에 주는지 확인

공식 매뉴얼 예제는 startYmd=endYmd(하루)만 보여줘서, 실제로 기간(예: 이틀치)을 넣으면 이틀치가 한번에 나오는지 아니면 하루치만 나오는지 확실치 않다. **본 수집 전에 반드시 이 셀로 먼저 확인**하고, 결과를 보고 다음 단계로 넘어갈지 판단할 것.

In [ ]:
probe_params = {
    'serviceKey': SERVICE_KEY,
    'pageNo': 1,
    'numOfRows': 5,
    'MobileOS': 'ETC',
    'MobileApp': 'BCcardContest',
    'startYmd': '20260101',
    'endYmd': '20260102',  # 이틀
    '_type': 'json',
}

resp = requests.get(API_URL, params=probe_params, timeout=20)
print('status_code:', resp.status_code)
data = resp.json()
print(data)

total = data['response']['body']['totalCount']
print()
print('totalCount:', total)
print('하루치가 대략 740개 안팎이었으니, 이게 약 1480(740*2)에 가까우면 기간쿼리가 정상 작동하는 것 — 여전히 740 근처면 기간이 무시되고 있는 것')


## 3. 본 수집 (2026-01-01 ~ 2026-06-30)

위 점검 결과에 따라 두 가지 버전 중 하나를 쓴다:
- **(A) 기간쿼리가 정상 작동하면** → 아래 3-A 셀 실행
- **(B) 기간쿼리가 안 되고 하루 단위로만 나오면** → 3-B 셀(181일 반복 요청)로 대체 — 개발계정 일 1,000건 한도 안에서 가능(181건)

둘 다 결과는 같은 형식의 `raw_rows` 리스트로 모인다.

### 3-A. 기간쿼리 한번에 (페이지네이션으로 전부 수집)

In [ ]:
raw_rows = []

page = 1
num_of_rows = 1000
total_count = None

while True:
    params = {
        'serviceKey': SERVICE_KEY,
        'pageNo': page,
        'numOfRows': num_of_rows,
        'MobileOS': 'ETC',
        'MobileApp': 'BCcardContest',
        'startYmd': '20260101',
        'endYmd': '20260630',
        '_type': 'json',
    }
    resp = requests.get(API_URL, params=params, timeout=30)
    body = resp.json()['response']['body']
    total_count = body['totalCount']
    items = body.get('items', {})
    item_list = items.get('item', []) if isinstance(items, dict) else []
    if isinstance(item_list, dict):  # 결과가 1건이면 dict로 올 수 있음
        item_list = [item_list]

    for it in item_list:
        if it.get('signguCode') in TARGET_REGIONS and it.get('touDivCd') == FOREIGNER_CODE:
            raw_rows.append(it)

    print(f'page {page} / 누적 {page*num_of_rows}/{total_count} / 후보지 매칭 누적 {len(raw_rows)}건')

    if page * num_of_rows >= total_count:
        break
    page += 1
    time.sleep(0.3)

print('수집 완료. 총', len(raw_rows), '건')


### 3-B. (대안) 하루씩 181번 반복 — 3-A가 기간쿼리를 지원 안 할 경우에만 사용

In [ ]:
import datetime

raw_rows_daily = []
start = datetime.date(2026, 1, 1)
end = datetime.date(2026, 6, 30)
d = start
call_count = 0

while d <= end:
    ymd = d.strftime('%Y%m%d')
    params = {
        'serviceKey': SERVICE_KEY,
        'pageNo': 1,
        'numOfRows': 1000,
        'MobileOS': 'ETC',
        'MobileApp': 'BCcardContest',
        'startYmd': ymd,
        'endYmd': ymd,
        '_type': 'json',
    }
    resp = requests.get(API_URL, params=params, timeout=30)
    call_count += 1
    try:
        body = resp.json()['response']['body']
        items = body.get('items', {})
        item_list = items.get('item', []) if isinstance(items, dict) else []
        if isinstance(item_list, dict):
            item_list = [item_list]
        for it in item_list:
            if it.get('signguCode') in TARGET_REGIONS and it.get('touDivCd') == FOREIGNER_CODE:
                raw_rows_daily.append(it)
    except Exception as e:
        print(ymd, '실패:', e)

    if call_count % 30 == 0:
        print(f'{ymd}까지 진행, 호출 {call_count}회, 매칭 {len(raw_rows_daily)}건')
    d += datetime.timedelta(days=1)
    time.sleep(0.2)

print('완료. 총 호출', call_count, '회 / 매칭', len(raw_rows_daily), '건')
raw_rows = raw_rows_daily  # 3-A 대신 이 결과를 이후 셀에서 사용


## 4. 정리 — 일별 → 월별 외국인 방문자수

In [ ]:
df = pd.DataFrame(raw_rows)
print('원본 행 수:', len(df))
df['STRD_YYMM'] = df['baseYmd'].str[:6]
df['touNum'] = df['touNum'].astype(float)
df['region'] = df['signguCode'].map(lambda c: TARGET_REGIONS[c][1])

monthly = df.groupby(['region', 'STRD_YYMM'])['touNum'].sum().unstack('region')
monthly = monthly.reindex(['202601','202602','202603','202604','202605','202606'])
monthly


## 5. 저장

In [ ]:
out_path = f'{BASE_DIR}/tour_datalab_foreign_visitors_monthly.csv'
monthly.to_csv(out_path, encoding='utf-8-sig')
print('저장 완료:', out_path)

# 원본 일별 데이터도 함께 저장 — 나중에 일별 패턴이 필요할 수도 있으니
raw_out_path = f'{BASE_DIR}/tour_datalab_foreign_visitors_daily_raw.csv'
df.to_csv(raw_out_path, encoding='utf-8-sig', index=False)
print('원본(일별) 저장 완료:', raw_out_path)


## 참고

- 개발계정은 일 1,000건 트래픽 제한 — 3-A(기간쿼리)가 되면 약 135콜, 3-B(일별)면 181콜로 둘 다 한도 안에 들어옴.
- `resultCode`가 `0000`이 아니면 에러 — 특히 `SERVICE_KEY_IS_NOT_REGISTERED_ERROR`가 뜨면 키 발급 후 약 30분 대기가 필요할 수 있음(공공데이터포털-관광공사 동기화 시간).
- `touDivCd`가 실제로는 문자열('3')로 오는지 숫자(3)로 오는지 응답을 보고 확인 필요 — 2번 사전점검 셀 결과에서 확인할 것(코드에는 문자열 비교로 짜뒀음).